# 🌊 밀물·썰물 관측소 이야기 — 조석 분석을 처음부터 끝까지

> 이 노트북 하나만 위에서부터 읽으면, "왜 조석 관측소를 따로 봐야 하는지"를 이해할 수 있도록 썼어요.
> 코드에는 주석을 촘촘히 달았고, 결과가 나올 때마다 **"그래서 무슨 뜻이냐"** 를 바로 아래에 풀어 적었어요.

### 한 줄 요약
**바다와 가까운 하천 관측소 15곳은 비가 아니라 밀물·썰물 때문에 하루 두 번씩 수위가 크게 오르내려요. 이 15곳은 데이터의 3%밖에 안 되는데, 오차의 절반을 만들어요.**

### 이야기 순서

| 장 | 무엇을 하나 | 쉽게 말하면 |
|---|---|---|
| 1 | 데이터와 타깃 이해 | "타깃이 정확히 뭘 재는 값이지?" |
| 2 | 이상한 관측소 발견 | "유독 튀는 애들이 있네?" |
| 3 | 자기상관이라는 도구 익히기 | "주기가 있는지 어떻게 숫자로 확인하지?" (가짜 데이터로 연습) |
| 4 | 진짜 데이터로 확인 | "15곳 **모두** 12.4시간 주기다" |
| 5 | 약한 조석과 24시간 관측소 | "조석은 있다/없다가 아니라 세기 차이다" |
| 6 | 조석이 분석을 어떻게 흐리나 | "조석을 빼고 보면 결론이 달라진다" |
| 7 | 모델은 조석을 맞히나 | "모델은 조석을 꽤 잘 맞힌다. 문제는 오히려 다른 데 있다" |
| 8 | 정리 | 모델링할 때 뭘 하면 되나 |

각 장에서 **몇 곳만 골라 예시를 보여주는 데서 끝내지 않고**, 같은 경향이 **15곳 전체**(또는 해당 집단 전체)에 나타나는지 꼭 확인했어요. 전체에 해당하지 않는 경우도 숨기지 않고 적었어요.

### 용어 사전 (처음 보면 여기부터)

| 용어 | 뜻 | 예시 |
|---|---|---|
| **타깃** (`target_maxrise_6h`) | 지금부터 6시간 안에 수위가 **최대 몇 m 더 올라가나** | 지금 1.0m, 앞으로 6시간 동안 최고 2.5m → 타깃 1.5m |
| **조석** | 달·해의 인력 때문에 바닷물이 하루 약 두 번 들어오고(밀물) 나가는(썰물) 현상. 주기는 약 **12시간 25분(12.42h)** | 인천 앞바다는 하루 두 번 수위가 8m 가까이 오르내려요 |
| **조석 관측소** | 바다와 가까워 하천 수위가 조석을 따라 움직이는 관측소 | 강 하구 근처 관측소 |
| **비조석** | 조석 관측소(와 고장 난 관측소 1곳)를 뺀 나머지 | 산골짜기 하천 관측소 |
| **dwl_1h** | 1시간 동안 수위가 얼마나 변했나 (`지금 수위 − 1시간 전 수위`) | +0.3이면 1시간 새 30cm 올랐다 |
| **자기상관** | 어떤 신호를 **자기 자신의 몇 시간 뒤 모습**과 비교한 상관계수. +1이면 똑같이 움직이고, −1이면 정반대로 움직여요 | 3장에서 그림으로 설명해요 |
| **제곱오차 합(SSE)** | (정답 − 예측)²을 모두 더한 것. RMSE는 이걸 행 수로 나누고 루트를 씌운 값이에요 | 큰 오차는 제곱되어서 훨씬 크게 반영돼요 |
| **사상** (`event_block`) | 데이터에서 시간이 끊기지 않고 이어진 한 덩어리(대체로 한 번의 호우 기간) | train에는 사상이 여러 개 있어요 |

## 0. 준비 — 라이브러리와 데이터 불러오기

- `cache/train_sorted.parquet`는 원본 train을 관측소·시간 순으로 정렬하고, 사상 번호(`event_block`)와 수위 변화(`dwl_*`) 열을 붙여 둔 파일이에요.
- 아래 셀은 그림 스타일과 **색 약속**을 정해요. 이 노트북 전체에서 색 뜻이 같아요.
  - 🟠 주황 = 조석 관측소 · 🔵 파랑 = 비조석(일반) · ⚪ 회색 = 전체 · 🟢 초록 = 약한 조석 · 🩷 분홍 = 24시간 주기 관측소

In [ ]:
import warnings
warnings.filterwarnings("ignore")          # 표준편차 0인 관측소 등에서 나오는 경고는 결과에 영향 없어서 숨김

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ---- 그림 기본 설정 (맥 한글 폰트) ----
plt.rcParams["font.family"] = ["DejaVu Sans", "AppleGothic"]   # 숫자는 DejaVu, 한글은 AppleGothic
plt.rcParams["axes.unicode_minus"] = False                    # 마이너스 기호 깨짐 방지
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": .6,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
                     "legend.frameon": False})

# ---- 경로: 이 노트북은 eda/ 폴더에 있다고 가정 ----
EDA = Path.cwd() if (Path.cwd() / "cache").exists() else Path.cwd().parent
CACHE = EDA / "cache"
TARGET = "target_maxrise_6h"

# ---- 색 약속 ----
C_ALL, C_NONTIDAL, C_TIDAL, C_WEAK, C_OPER = "#9a9890", "#2a78d6", "#eb6834", "#1baf7a", "#e87ba4"
C_RAIN = "#eda100"

cols = ["station_id", "station_code", "hour", "event_block", "pos_in_block",
        "wl_t", "dwl_1h", "dwl_3h", "dwl_6h", TARGET,
        "imerg_rain_6h", "sfc_rain_1h", "sfc_rain_6h"]
df = pd.read_parquet(CACHE / "train_sorted.parquet", columns=cols)
df["station_id"] = df["station_id"].astype(str)
print(f"행 {len(df):,}개 · 관측소 {df.station_id.nunique()}곳 · 사상 {df.event_block.nunique()}개")

### 도우미 함수 3개

분석 내내 쓰는 함수예요. 핵심 아이디어는 **"같은 관측소의 L시간 뒤 값을 옆에 붙이기"** 예요.

- `shift_lookup`: 각 행 옆에 "같은 관측소, L시간 뒤" 값을 붙여요. 그 시각 데이터가 없으면(사상이 끊긴 곳) NaN이 들어가서, 서로 다른 사상이 섞이지 않아요.
- `station_corr`: 관측소마다 두 열의 상관계수를 한꺼번에 구해요. 465곳을 반복문으로 돌리면 느려서, 합계만 모아 한 번에 계산해요.
- `symlog_x`: 0 근처는 선형, 멀리는 로그로 그리는 x축이에요. 수위 변화처럼 0 근처에 몰리고 가끔 크게 튀는 값을 볼 때 써요.

In [ ]:
def keyed(d):
    """(관측소, 시각)을 정수 하나로 합친 키. 예: 관측소코드 12, 시각 345 → 12_000_345"""
    return d["station_code"].to_numpy(np.int64) * 1_000_000 + d["hour"].to_numpy(np.int64)

def shift_lookup(d, key, col, dh):
    """각 행에 대해 '같은 관측소의 (지금 + dh)시간' col 값을 가져온다. 없으면 NaN."""
    s = pd.Series(d[col].to_numpy(np.float64), index=key)
    return s.reindex(key + dh).to_numpy()

def station_corr(d, x, y, min_n=50):
    """관측소별 피어슨 상관계수. 합·제곱합을 groupby로 모아 한 번에 계산(빠름). 표본이 50개 이하면 NaN."""
    t = pd.DataFrame({"s": d["station_id"].to_numpy(), "x": x, "y": y}).dropna()
    t["xy"], t["xx"], t["yy"] = t.x * t.y, t.x * t.x, t.y * t.y
    g = t.groupby("s").agg(n=("x", "size"), sx=("x", "sum"), sy=("y", "sum"),
                           sxy=("xy", "sum"), sxx=("xx", "sum"), syy=("yy", "sum"))
    cov = g.sxy / g.n - (g.sx / g.n) * (g.sy / g.n)        # 공분산 = E[xy] − E[x]E[y]
    vx = g.sxx / g.n - (g.sx / g.n) ** 2                     # 분산
    vy = g.syy / g.n - (g.sy / g.n) ** 2
    return (cov / np.sqrt(vx * vy)).where(g.n > min_n)

def symlog_x(ax, linthresh=0.1):
    ax.set_xscale("symlog", linthresh=linthresh)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))

key = keyed(df)

## 1. 타깃이 정확히 뭘 재는 값일까?

조석 얘기를 하기 전에, 우리가 맞혀야 하는 **타깃**부터 확실히 알고 가요. 대회 설명대로라면 이렇게 계산돼요.

> **타깃 = (앞으로 1~6시간 동안의 수위 중 가장 높은 값) − (지금 수위)**

진짜 그런지 데이터로 직접 다시 계산해서 맞춰 볼게요. 6시간 뒤까지 데이터가 다 있는 행만 비교할 수 있어요.

In [ ]:
# 같은 관측소의 1~6시간 뒤 수위를 옆에 붙인다 → (행 수 × 6) 행렬
future = np.column_stack([shift_lookup(df, key, "wl_t", k) for k in range(1, 7)])
has_all6 = np.isfinite(future).all(axis=1)                     # 6시간이 다 있는 행만

recomputed = future[has_all6].max(axis=1) - df["wl_t"].to_numpy()[has_all6]
given = df[TARGET].to_numpy()[has_all6]
same = np.isclose(recomputed, given, atol=1e-3)
print(f"비교 가능한 행: {has_all6.mean():.0%}")
print(f"직접 계산한 값과 타깃이 같은 비율: {same.mean():.1%}")

100% 일치해요. 그러니까 타깃은 **"지금부터 6시간 안에 물이 최대 얼마나 더 차오르나"** 예요.

이 정의에서 중요한 점이 하나 있어요. **비가 안 와도, 6시간 안에 물이 한 번이라도 높아지기만 하면 타깃은 커져요.** 이게 조석 관측소 이야기의 출발점이에요. 그림으로 한 번 볼게요.

> ⚠️ **누수 주의 — 이 계산은 '미래 행'을 썼어요.**
> 타깃 정의를 확인하려고 같은 관측소의 1~6시간 **뒤** 수위를 가져다 썼어요. EDA에서 정의를 확인하는 용도로는 괜찮아요.
> 그런데 **test 파일에도 같은 관측소의 뒤 시각 행이 들어 있어요**(test 행의 61%는 뒤 6시간이 전부 있어요). 같은 방법을 쓰면 test 타깃을 거의 그대로 계산할 수 있어요.
> 이게 이 대회에서 가장 큰 누수 통로예요. 그래서 **모델 입력에는 절대 쓰지 않고**, 이 노트북의 모델(7장)도 과거 값만 써요. 8장 끝에 노트북 전체의 누수 점검표를 붙여 뒀어요.

In [ ]:
# 타깃이 가장 큰 행 하나를 골라 그 주변 수위를 그려 본다
i = df[TARGET].idxmax()
row = df.loc[i]
s = df[(df.station_id == row.station_id) & (df.event_block == row.event_block)].sort_values("hour")
t0 = row.hour
win = s[(s.hour >= t0 - 12) & (s.hour <= t0 + 12)]

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(win.hour - t0, win.wl_t, "-o", ms=3, color=C_NONTIDAL, label="수위")
ax.axvspan(0.5, 6.5, color=C_RAIN, alpha=.15, label="앞으로 6시간 (타깃을 재는 구간)")
fut = win[(win.hour > t0) & (win.hour <= t0 + 6)]
peak = fut.loc[fut.wl_t.idxmax()]
ax.annotate("", xy=(peak.hour - t0, peak.wl_t), xytext=(peak.hour - t0, row.wl_t),
            arrowprops=dict(arrowstyle="<->", color=C_TIDAL, lw=2))
ax.axhline(row.wl_t, color="k", lw=.6, ls=":")
ax.text(peak.hour - t0 + .3, (peak.wl_t + row.wl_t) / 2, f"타깃 = {row[TARGET]:.2f} m", color=C_TIDAL)
ax.scatter([0], [row.wl_t], s=80, color="k", zorder=5, label="지금 (t)")
ax.set_xlabel("지금으로부터 몇 시간 (h)"); ax.set_ylabel("수위 (m)")
ax.set_title(f"예시: {row.station_id}에서 타깃이 가장 컸던 순간")
ax.legend(loc="upper left", fontsize=8); plt.show()

## 2. 발견 — "유독 튀는 관측소들이 있다"

### 2-1. 오차는 어디서 오나?

타깃은 대부분 0 근처예요(중앙값 1cm). 그런데 가끔 몇 m씩 튀어요. RMSE는 오차를 **제곱**해서 더하니까, 이렇게 튀는 몇 개가 점수를 거의 다 결정해요.

모든 예측을 0으로 찍었다고 치고(= 가장 단순한 모델), 오차가 어디서 오는지 봐요.

In [ ]:
y = df[TARGET].to_numpy(float)
e2 = y ** 2                                               # 0으로 예측했을 때의 제곱오차
top1 = e2 >= np.quantile(e2, .99)                         # 제곱오차 상위 1% 행
print(f"타깃 평균 {y.mean():.3f} m · 중앙값 {np.median(y):.3f} m · 최대 {y.max():.2f} m")
print(f"상위 1% 행이 전체 제곱오차에서 차지하는 비중: {e2[top1].sum() / e2.sum():.0%}")

**행 1%가 오차의 2/3를 만들어요.** 그럼 그 1%는 어디서 나올까요? 관측소 단위로 쪼개 봐요.

아래 왼쪽 그림은 관측소 465곳을 **오차가 큰 순서로 줄 세우고** 누적 비중을 그린 거예요. 만약 모든 관측소가 비슷하게 기여한다면 점선(대각선)처럼 나와야 해요.

In [ ]:
# 관측소 단위 요약표: 타깃 평균·표준편차, 0 예측 제곱오차 합과 비중
st = df.groupby("station_id").agg(tmean=(TARGET, "mean"), tstd=(TARGET, "std"),
                                  sse=(TARGET, lambda v: float((v.astype(float) ** 2).sum())), n=(TARGET, "size"))
st["sse_share"] = st.sse / st.sse.sum()

order = st.sort_values("sse_share", ascending=False)
cum = order.sse_share.cumsum().to_numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(np.arange(1, len(cum) + 1), cum * 100, color=C_NONTIDAL)
ax[0].plot([0, len(cum)], [0, 100], ":", color=C_ALL, label="모두 똑같이 기여한다면")
ax[0].axvline(15, color=C_TIDAL, ls="--", lw=1)
ax[0].text(18, cum[14] * 100 - 8, f"상위 15곳 = {cum[14]:.0%}", color=C_TIDAL)
ax[0].set_xlabel("오차 큰 순서로 줄 세운 관측소 수"); ax[0].set_ylabel("누적 제곱오차 비중 (%)")
ax[0].set_title("몇 곳이 오차를 독차지한다"); ax[0].legend(loc="lower right", fontsize=8)

ax[1].scatter(st.tstd, st.tmean, s=12, color=C_NONTIDAL, alpha=.7)
ax[1].set_xscale("log"); ax[1].set_xlabel("관측소별 타깃 표준편차 (m, log)"); ax[1].set_ylabel("관측소별 타깃 평균 (m)")
ax[1].set_title("관측소 465곳 — 오른쪽 위로 떨어져 나온 무리")
fig.tight_layout(); plt.show()

**뭐가 보이나요?**
- 왼쪽: 465곳 중 **상위 15곳만으로 오차의 절반 정도**가 나와요.
- 오른쪽: 대부분 관측소는 왼쪽 아래에 몰려 있는데, 몇 곳은 **평균이 1~2m, 표준편차가 1~3m**로 멀리 떨어져 있어요.

여기서 이상한 점이 있어요. 타깃 **평균이 2m**라는 건 "아무 때나 재도 6시간 안에 평균 2m씩 물이 차오른다"는 뜻이에요. 비 때문이라면 말이 안 돼요. 대부분의 시간은 비가 안 오니까요. 이 관측소들의 수위를 직접 봐요.

### 2-2. 튀는 관측소들의 수위를 직접 보면

In [ ]:
# 데이터가 가장 긴 사상 하나를 골라, 표준편차 상위 3곳의 수위와 비를 함께 그린다
long_block = df.groupby("event_block").hour.nunique().idxmax()
ex = st.sort_values("tstd", ascending=False).index[:3].tolist()

fig, axes = plt.subplots(len(ex), 1, figsize=(11, 2.2 * len(ex)), sharex=True)
for ax, s_ in zip(axes, ex):
    d = df[(df.station_id == s_) & (df.event_block == long_block)].sort_values("hour")
    ax.plot(d.pos_in_block, d.wl_t, color=C_TIDAL)
    a2 = ax.twinx()                                        # 오른쪽 축: 비 (위에서 아래로 내려오게 뒤집음)
    a2.bar(d.pos_in_block, d.sfc_rain_1h, color=C_RAIN, alpha=.5, width=1)
    a2.invert_yaxis(); a2.grid(False); a2.set_ylabel("비 (mm)", fontsize=8)
    ax.set_ylabel("수위 (m)"); ax.set_title(s_, fontsize=9)
axes[-1].set_xlabel(f"가장 긴 사상 #{long_block} 시작 후 시간 (h)")
fig.suptitle("표준편차 상위 3곳: 비(위에서 내려오는 막대)와 상관없이 톱니처럼 오르내린다", x=.02, ha="left")
fig.tight_layout(); plt.show()

**톱니 모양**이에요. 하루에 두 번 정도, 비가 오든 안 오든 3~7m를 오르내려요.

💡 **가설: 이 관측소들은 바다와 가까워서 밀물·썰물(조석)을 따라 움직인다.**

조석이라면 타깃이 큰 이유도 설명돼요. 썰물로 물이 가장 낮을 때 재면, 앞으로 6시간 안에 반드시 밀물이 들어오니까 타깃이 몇 m씩 나와요.

그런데 "그림 3장 보니 톱니네"는 증거가 약해요. 그래서 다음 두 가지를 할 거예요.
1. **숫자로** 주기를 재는 방법을 정해요 (3장).
2. 그 방법으로 465곳을 **전부** 검사해요 (4장).

## 3. 도구 익히기 — 자기상관으로 주기 찾기 (가짜 데이터로 연습)

### 3-1. 자기상관이 뭔가요?

**신호를 몇 시간 밀어서 원래 신호와 겹쳐 본다**고 생각하면 돼요.

- 12.4시간마다 반복되는 조석을 **12시간** 밀면? 거의 같은 모양이 겹쳐요 → 상관 **+1에 가까움**
- **6시간**(반 주기) 밀면? 밀물 자리에 썰물이 와요 → 상관 **−1에 가까움**
- 주기가 없는 신호는 몇 시간만 밀어도 상관이 **0**으로 떨어져요.

실제 데이터에 쓰기 전에, **정답을 아는 가짜 조석**을 만들어서 이 성질을 눈으로 확인해 볼게요.

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(24 * 6)                                   # 6일치, 1시간 간격
TIDE_PERIOD = 12.42                                     # 반일주조(M2) 주기
fake_wl = 2.0 * np.sin(2 * np.pi * t / TIDE_PERIOD) + rng.normal(0, .1, len(t))   # 진폭 2m + 약간의 잡음
fake_dwl = np.r_[np.nan, np.diff(fake_wl)]              # 1시간 수위 변화 = dwl_1h

def acf_1d(x, L):
    """한 줄짜리 시계열의 시차 L 자기상관"""
    a, b = x[:-L], x[L:]
    m = np.isfinite(a) & np.isfinite(b)
    return np.corrcoef(a[m], b[m])[0, 1]

fig, ax = plt.subplot_mosaic([["wave", "wave", "wave"], ["s6", "s12", "acf"]], figsize=(12, 6.5))
# (위) 가짜 조석 + 6h/12h 밀어 본 모습
w = t < 48
ax["wave"].plot(t[w], fake_wl[w], color=C_TIDAL, lw=2, label="원래 수위")
ax["wave"].plot(t[w] + 6, fake_wl[w], color=C_NONTIDAL, lw=1, ls="--", label="6시간 민 것 → 반대로 움직임")
ax["wave"].plot(t[w] + 12, fake_wl[w], color=C_WEAK, lw=1, ls=":", label="12시간 민 것 → 거의 겹침")
ax["wave"].set_xlim(0, 48); ax["wave"].set_xlabel("시간 (h)"); ax["wave"].set_ylabel("수위 (m)")
ax["wave"].legend(fontsize=8, ncol=3, loc="upper right"); ax["wave"].set_title("가짜 조석 (주기 12.42시간)")
# (아래 왼쪽·가운데) 산점도로 보는 상관
for k, L, col in [("s6", 6, C_NONTIDAL), ("s12", 12, C_WEAK)]:
    ax[k].scatter(fake_dwl[:-L], fake_dwl[L:], s=6, color=col)
    ax[k].set_xlabel("지금의 dwl_1h"); ax[k].set_ylabel(f"{L}시간 뒤의 dwl_1h")
    ax[k].set_title(f"시차 {L}h → 상관 {acf_1d(fake_dwl, L):+.2f}")
# (아래 오른쪽) 시차 1~26h 전체 = 자기상관 함수
lags = np.arange(1, 27)
ax["acf"].plot(lags, [acf_1d(fake_dwl, L) for L in lags], "-o", ms=3, color=C_TIDAL)
ax["acf"].axhline(0, color="k", lw=.5)
for L, lab in [(6.21, "반 주기"), (12.42, "한 주기")]:
    ax["acf"].axvline(L, color=C_ALL, ls=":"); ax["acf"].text(L + .3, .8, lab, fontsize=8)
ax["acf"].set_xlabel("시차 (h)"); ax["acf"].set_ylabel("자기상관"); ax["acf"].set_title("자기상관 곡선: 파도 모양")
fig.tight_layout(); plt.show()

**정리**: 조석이면 자기상관 곡선이 **6시간에서 골(−), 12시간에서 봉우리(+)** 를 그리는 파도 모양이 돼요. 이게 우리가 찾을 **조석의 지문**이에요.

### 3-2. 왜 수위(wl)가 아니라 수위 변화(dwl_1h)로 보나요?

실제 하구에서는 조석 위에 **홍수로 몇 m 차오르는 흐름**이 같이 얹혀요. 수위 자체로 자기상관을 재면, 이 큰 흐름 때문에 **어느 시차에서나 상관이 높게 나와서** 주기가 묻혀요.
1시간 차분(`dwl_1h`)을 쓰면 느린 흐름은 거의 사라지고 주기만 남아요. 가짜 데이터로 비교해 볼게요.

같이 **보통 하천**(비가 오면 불었다가 서서히 빠지는 하천)도 가짜로 만들어서, 주기가 없으면 곡선이 어떻게 생기는지도 봐 둘게요.

In [ ]:
# (1) 조석 + 3일에 걸쳐 4m 차오르는 홍수
flood = 4 / (1 + np.exp(-(t - 72) / 10))                 # 느린 S자 상승
tide_flood = fake_wl + flood
tf_dwl = np.r_[np.nan, np.diff(tide_flood)]

# (2) 보통 하천: 비가 드문드문 오고, 수위는 비에 반응해 올랐다가 천천히 빠짐
rain = np.where(rng.random(len(t)) < .05, rng.gamma(2, 5, len(t)), 0)
unit = (np.arange(24) ** 2) * np.exp(-np.arange(24) / 2.5)   # 단위도: 비 1mm가 수위에 퍼지는 모양
river = np.convolve(rain, unit / unit.sum() * .1)[:len(t)]
river_dwl = np.r_[np.nan, np.diff(river)]

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(t, tide_flood, color=C_TIDAL, label="조석 + 홍수")
ax[0].plot(t, river + 1, color=C_NONTIDAL, label="보통 하천 (+1 띄움)")
ax[0].set_xlabel("시간 (h)"); ax[0].set_ylabel("수위 (m)"); ax[0].legend(fontsize=8); ax[0].set_title("가짜 수위 두 개")
ax[1].plot(lags, [acf_1d(tide_flood, L) for L in lags], "-o", ms=3, color=C_TIDAL, label="조석+홍수")
ax[1].plot(lags, [acf_1d(river, L) for L in lags], "-o", ms=3, color=C_NONTIDAL, label="보통 하천")
ax[1].set_title("수위(wl)로 잰 자기상관 → 둘 다 높아서 구분이 안 됨")
ax[2].plot(lags, [acf_1d(tf_dwl, L) for L in lags], "-o", ms=3, color=C_TIDAL, label="조석+홍수")
ax[2].plot(lags, [acf_1d(river_dwl, L) for L in lags], "-o", ms=3, color=C_NONTIDAL, label="보통 하천")
ax[2].set_title("수위 변화(dwl_1h)로 잰 자기상관 → 차이가 확실")
for a in ax[1:]:
    a.axhline(0, color="k", lw=.5); a.set_xlabel("시차 (h)"); a.set_ylim(-1, 1.05); a.legend(fontsize=8)
fig.tight_layout(); plt.show()

- 가운데: 수위로 재면 조석이든 아니든 곡선이 높게 깔려서 **구분이 어려워요**.
- 오른쪽: 수위 변화로 재면 조석은 **파도 모양**, 보통 하천은 **몇 시간 만에 0으로 떨어지는 모양**이에요.

그래서 실제 판정 기준은 이렇게 정했어요 (원본 `src/s04_hydro.py`).

> **조석 관측소 = `dwl_1h` 자기상관이 (12h 또는 13h에서 0.6 초과) 그리고 (6h 또는 7h에서 음수)**

- 12.42시간은 정수 시차 12와 13 사이에 걸려서 둘 중 큰 값을 쓰고, 반 주기(6.21h)도 같은 이유로 6과 7 중 작은 값을 써요.
- 0.6은 "파도가 뚜렷하다"고 볼 만한 문턱이에요. 이 문턱이 임의적이라는 문제는 5장에서 다시 다뤄요.

## 4. 진짜 데이터로 확인 — 465곳 전부 검사

### 4-1. 모든 관측소의 자기상관 계산

In [ ]:
x_now = df["dwl_1h"].to_numpy(np.float64)
# 시차 1~26시간마다, 관측소별로 corr(지금 dwl_1h, L시간 뒤 dwl_1h)
A = pd.DataFrame({L: station_corr(df, x_now, shift_lookup(df, key, "dwl_1h", -L)) for L in range(1, 27)})
# A: 행 = 관측소 465곳, 열 = 시차 1..26h

st["acf1"] = A[1]
st["acf12"] = A[[12, 13]].max(axis=1)     # 한 주기 근처의 봉우리
st["acf6"] = A[[6, 7]].min(axis=1)        # 반 주기 근처의 골
st["acf24"] = A[24]

TIDAL = st[(st.acf12 > 0.6) & (st.acf6 < 0)].sort_values("sse_share", ascending=False).index.tolist()
st["is_tidal"] = st.index.isin(TIDAL)
print(f"조석 관측소로 판정된 곳: {len(TIDAL)}곳")
print(TIDAL)
A.head(3).round(2)

**15곳**이 판정됐어요. 먼저 **예시 두 곳**(조석 1곳, 평범한 관측소 1곳)의 곡선을 3장의 가짜 데이터 곡선과 나란히 놓아 봐요.

In [ ]:
normal_ex = (st[~st.is_tidal].acf1 - st[~st.is_tidal].acf1.median()).abs().idxmin()   # 지속성이 딱 중간인 평범한 곳
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
ax[0].plot(lags, [acf_1d(fake_dwl, L) for L in lags], ":", color=C_ALL, label="3장 가짜 조석")
ax[0].plot(A.columns, A.loc[TIDAL[0]], "-o", ms=3, color=C_TIDAL, label=f"{TIDAL[0]} (실제)")
ax[0].set_title("조석 관측소 예시: 가짜 조석과 거의 똑같다")
ax[1].plot(lags, [acf_1d(river_dwl, L) for L in lags], ":", color=C_ALL, label="3장 가짜 보통 하천")
ax[1].plot(A.columns, A.loc[normal_ex], "-o", ms=3, color=C_NONTIDAL, label=f"{normal_ex} (실제)")
ax[1].set_title("평범한 관측소 예시: 금방 0으로")
for a in ax:
    a.axhline(0, color="k", lw=.5); a.set_xlabel("시차 (h)"); a.legend(fontsize=8)
ax[0].set_ylabel("dwl_1h 자기상관"); fig.tight_layout(); plt.show()

예시 두 곳은 예상대로예요. 하지만 **두 곳만 보고 결론 내리면 안 돼요.** 이제 465곳 **전부**를 한 그림에 그려요.
- 주황 선 15개 = 조석 판정된 곳 **하나하나**
- 파란 띠 = 나머지 450곳의 가운데 80%(10~90% 구간), 진한 파란 선 = 중앙값

In [ ]:
nt = A.loc[~st.is_tidal]
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.fill_between(A.columns, nt.quantile(.1), nt.quantile(.9), color=C_NONTIDAL, alpha=.2, label="나머지 450곳의 10~90%")
ax.plot(A.columns, nt.median(), color=C_NONTIDAL, lw=2, label="나머지 450곳 중앙값")
for s_ in TIDAL:
    ax.plot(A.columns, A.loc[s_], color=C_TIDAL, lw=1, alpha=.8)
ax.plot([], [], color=C_TIDAL, label="조석 15곳 (한 줄 = 한 관측소)")
for L in (6.21, 12.42, 18.63, 24.84):
    ax.axvline(L, color=C_ALL, ls=":", lw=.8)
ax.axhline(0, color="k", lw=.5); ax.set_xlabel("시차 (h)  · 점선 = 조석 반 주기의 배수"); ax.set_ylabel("dwl_1h 자기상관")
ax.set_title("465곳 전부: 15곳 모두 같은 파도, 나머지는 모두 금방 0")
ax.legend(fontsize=8, loc="lower left"); plt.show()

**15곳 모두** 6h·18h에서 골, 12h·24h에서 봉우리를 그려요. 나머지 450곳의 띠는 3시간쯤이면 0 근처로 붙어요.

### 4-2. 15곳 하나하나가 기준을 어떻게 통과하나

판정 기준 두 개를 두 축으로 놓고 465곳을 전부 찍어요. 오른쪽 아래 주황 상자가 "조석 구역"이에요.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3), gridspec_kw={"width_ratios": [1, 1.3]})
ax[0].add_patch(plt.Rectangle((0.6, -1), 0.4, 1, color=C_TIDAL, alpha=.1))
ax[0].scatter(st.acf12[~st.is_tidal], st.acf6[~st.is_tidal], s=8, color=C_NONTIDAL, alpha=.5, label="나머지")
ax[0].scatter(st.acf12[st.is_tidal], st.acf6[st.is_tidal], s=30, color=C_TIDAL, label="조석 15곳")
ax[0].axvline(.6, color=C_TIDAL, ls="--", lw=.8); ax[0].axhline(0, color=C_TIDAL, ls="--", lw=.8)
ax[0].set_xlabel("12~13h 자기상관 (봉우리)"); ax[0].set_ylabel("6~7h 자기상관 (골)")
ax[0].set_title("판정 기준 평면"); ax[0].legend(fontsize=8)

# 15곳 각각의 봉우리/골 값 막대
tt = st.loc[TIDAL]
xx = np.arange(len(tt))
ax[1].bar(xx - .2, tt.acf12, .4, color=C_TIDAL, label="12~13h 봉우리 (> 0.6 이어야)")
ax[1].bar(xx + .2, tt.acf6, .4, color=C_NONTIDAL, label="6~7h 골 (< 0 이어야)")
ax[1].axhline(.6, color=C_TIDAL, ls="--", lw=.8); ax[1].axhline(0, color="k", lw=.5)
ax[1].set_xticks(xx); ax[1].set_xticklabels(tt.index, rotation=60, fontsize=7)
ax[1].set_title("15곳 전부 두 조건을 통과 (왼쪽일수록 오차 기여 큼)"); ax[1].legend(fontsize=7, loc="lower left")
fig.tight_layout(); plt.show()

### 4-3. 주기가 정말 12.4시간인가? (15곳 각각 측정)

"12시간 근처에서 봉우리"만으로는 12시간인지 12.4시간인지 모르죠. 조석이 맞다면 주기가 **12.42시간**에 가까워야 해요.
시차는 1시간 단위라서, 봉우리 양옆 세 점에 포물선을 맞춰 **소수점까지** 봉우리 위치를 추정해요.

In [ ]:
def peak_period(acf_row, lo=10, hi=16):
    """시차 lo~hi 사이 최댓값 주변 3점에 포물선을 맞춰 봉우리 위치(=주기)를 소수점으로 추정"""
    a = acf_row.to_numpy(); lg = acf_row.index.to_numpy()
    i = np.argmax(a[lo - 1:hi]) + lo - 1                  # 열 index (시차 1 → 0)
    y0, y1, y2 = a[i - 1], a[i], a[i + 1]
    return lg[i] + 0.5 * (y0 - y2) / (y0 - 2 * y1 + y2)

st["period"] = A.apply(peak_period, axis=1)
p = st.loc[TIDAL, "period"]
print(f"조석 15곳 추정 주기: 최소 {p.min():.2f}h · 중앙값 {p.median():.2f}h · 최대 {p.max():.2f}h")

fig, ax = plt.subplots(figsize=(10, 3.3))
ax.scatter(range(len(p)), p, s=40, color=C_TIDAL, zorder=3)
ax.axhline(12.42, color="k", ls="--", lw=.8); ax.text(len(p) - .5, 12.45, "달 반일주조 M2 = 12.42h", ha="right", fontsize=8)
ax.axhline(12.0, color=C_ALL, ls=":", lw=.8); ax.text(len(p) - .5, 11.95, "태양 반일주조 S2 = 12.00h", ha="right", va="top", fontsize=8)
ax.set_xticks(range(len(p))); ax.set_xticklabels(p.index, rotation=60, fontsize=7)
ax.set_ylim(11.7, 12.7); ax.set_ylabel("추정 주기 (h)"); ax.set_title("15곳 모두 주기 12.2~12.4시간")
plt.show()

15곳 **모두 12.2~12.4시간**이에요. 12.42시간(달이 만드는 조석)에 가깝고, 12.00시간(해가 만드는 조석) 쪽으로 살짝 당겨져 있어요. 실제 바다에서는 이 두 조석이 섞이니까 자연스러운 결과예요.
**우연히 12시간 근처에 봉우리가 생긴 게 아니라, 진짜 조석 주기**라는 뜻이에요.

### 4-4. 15곳 전부의 실제 수위 모습

2장에서는 3곳만 봤어요. 이번엔 **15곳 전부**를 같은 사상 구간에서 그려요(칸마다 y축 범위가 달라요).

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(13, 10), sharex=True)
for ax, s_ in zip(axes.ravel(), TIDAL):
    d = df[df.station_id == s_]
    blk = d.groupby("event_block").size().idxmax()          # 그 관측소에서 가장 긴 사상
    d = d[d.event_block == blk].sort_values("hour").iloc[:96]  # 앞 4일만
    ax.plot(d.pos_in_block - d.pos_in_block.min(), d.wl_t, color=C_TIDAL, lw=1)
    a2 = ax.twinx(); a2.bar(d.pos_in_block - d.pos_in_block.min(), d.sfc_rain_1h, color=C_RAIN, alpha=.5, width=1)
    a2.invert_yaxis(); a2.set_ylim(max(d.sfc_rain_1h.max() * 3, 5), 0); a2.grid(False); a2.set_yticks([])
    rng_m = d.wl_t.max() - d.wl_t.min()
    ax.set_title(f"{s_}  (변동폭 {rng_m:.1f} m, 주기 {st.loc[s_, 'period']:.2f} h)", fontsize=8)
    ax.tick_params(labelsize=7)
for a in axes[-1]: a.set_xlabel("시간 (h)", fontsize=8)
fig.suptitle("조석 15곳 전부 (각자 가장 긴 사상의 앞 4일 · 막대 = 비)", x=.02, ha="left")
fig.tight_layout(); plt.show()

15곳 **모두** 톱니 모양이에요. 자세히 보면 세 가지가 더 보여요.

1. **세기가 제각각**: 제목의 '변동폭'은 4일 동안의 최고−최저예요. ST0358처럼 조석만으로 7m를 오가는 곳도 있고, ST0130·ST0123처럼 1m 안팎인 곳도 있어요. 바다와의 거리, 강폭에 따라 조석이 전해지는 세기가 다르기 때문이에요.
2. **톱니가 비대칭인 곳이 많아요**: ST0347, ST0313, ST0159, ST0129 등은 **확 올랐다가(밀물) 천천히 내려가요(썰물)**. 강 하구에서 흔한 모양이고, 6-1절에서 다시 중요하게 나와요.
3. **홍수 위에 조석이 얹혀요**: ST0129, ST0192, ST0051, ST0164는 비(막대)가 온 뒤 수위가 전체적으로 몇 m 오르는데, 그 위에서도 톱니가 계속돼요. 3-2절에서 가짜 데이터로 본 '조석 + 홍수' 상황이 실제로 있는 거예요. 그래서 이런 곳의 변동폭에는 홍수 상승분도 섞여 있어요.

오차 기여는 변동폭이 큰 곳일수록 커요(줄 세운 순서가 오차 기여 순서예요).

In [ ]:
# 관측소 안에서 '6시간 누적 강우의 순위'와 '타깃의 순위'의 상관 (스피어만과 같은 원리)
rain6 = np.maximum(df.imerg_rain_6h, df.sfc_rain_6h)            # 위성·지상 중 큰 값
rr = rain6.groupby(df.station_id).rank(pct=True).to_numpy()
yr = df[TARGET].groupby(df.station_id).rank(pct=True).to_numpy()
st["rain_rho"] = station_corr(df, rr, yr)

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.hist(st.rain_rho[~st.is_tidal].dropna(), bins=40, color=C_NONTIDAL, alpha=.6, label="나머지 관측소")
for v in st.loc[TIDAL, "rain_rho"]:
    ax.axvline(v, color=C_TIDAL, lw=1.2, alpha=.8)
ax.plot([], [], color=C_TIDAL, label="조석 15곳 (선 하나 = 한 곳)")
ax.set_xlabel("강우–타깃 순위 상관 ρ"); ax.set_ylabel("관측소 수"); ax.legend(fontsize=8)
ax.set_title("조석 15곳은 전부 ρ ≈ 0: 비가 와도 타깃이 안 커진다")
plt.show()
lo_nt = (st.rain_rho[~st.is_tidal] <= st.loc[TIDAL, 'rain_rho'].max()).sum()
print(f"조석 15곳 ρ 범위: {st.loc[TIDAL, 'rain_rho'].min():+.2f} ~ {st.loc[TIDAL, 'rain_rho'].max():+.2f}")
print(f"나머지 중앙값: {st.rain_rho[~st.is_tidal].median():+.2f} · 나머지 중 ρ가 조석 최댓값 이하인 곳: {lo_nt}곳")

조석 15곳은 **전부 ρ가 −0.03~+0.12**로, 비와 거의 무관해요. 나머지 관측소의 중앙값은 +0.42예요.
(나머지 450곳 중에도 ρ가 이만큼 낮은 곳이 24곳 있어요. 5장에서 볼 약한 조석·24시간 관측소, 그리고 물이 거의 안 움직이는 곳들이에요.)

### 4-6. 그래서 타깃은 언제 커지나? — "썰물로 한참 내려와 있을 때" (15곳 전부)

조석 가설이 맞다면 **썰물로 물이 한참 빠져 있을 때 재면 타깃이 크고**, 밀물로 한참 차 있을 때 재면 작아야 해요. 곧 반대 방향으로 움직일 테니까요.

#### ⚠️ 여기서 빠지기 쉬운 함정: 미래가 섞인 기준
처음에는 "지금 수위가 **그 사상 전체**에서 몇 번째로 낮은가"로 물 높이를 쟀어요. 그런데 사상 전체에는 **지금 이후의 수위도** 들어 있어요.
그러니까 "사상에서 가장 낮은 순간"을 고르면, **정의상 그 뒤에는 물이 더 높을 수밖에 없어요.** 조석이 아니어도 타깃이 크게 나오는 거죠. 이런 걸 **누수**라고 해요. 답을 살짝 보고 나서 "봐, 맞지?" 하는 셈이에요.

그래서 **과거 정보만 쓰는 기준**으로 바꿨어요.
- `dwl_6h = 지금 수위 − 6시간 전 수위`: 모델 입력에도 있는 값이고 과거만 봐요.
- 6시간은 조석 반 주기(6.2h)라서, 이 값이 크게 음수면 **"썰물로 한참 내려왔다 → 곧 밀물"**, 크게 양수면 **"밀물로 한참 올라왔다 → 곧 썰물"** 이에요.
- 관측소마다 조차가 다르니까, 관측소 안에서 `dwl_6h`를 10칸(0 = 가장 많이 내려옴, 9 = 가장 많이 올라옴)으로 나눠요.

아래 첫 그림에서 두 기준을 나란히 놓고 함정을 먼저 확인해요.

In [ ]:

# ❌ 누수 있는 기준: 사상 '전체'(지금 이후 포함)에서 지금 수위의 순위 (0 = 가장 낮음)
df["wl_rank_leak"] = df.groupby(["station_id", "event_block"]).wl_t.rank(pct=True)
gap_leak = (df[df.wl_rank_leak <= .1].groupby("station_id")[TARGET].mean()
            - df[df.wl_rank_leak >= .9].groupby("station_id")[TARGET].mean())

# ✅ 과거만 쓰는 기준: 지난 6시간 수위 변화를 관측소 안에서 10칸으로 (0 = 가장 많이 내려옴)
df["d6_dec"] = np.minimum((df.groupby("station_id").dwl_6h.rank(pct=True) * 10).astype(int), 9)
curve = df.groupby(["station_id", "d6_dec"])[TARGET].mean().unstack()
gap = curve[0] - curve[9]            # (가장 많이 내려와 있을 때 − 가장 많이 올라와 있을 때) 타깃 평균, m

is_gen = st.index[(~st.is_tidal) & (st.index != "ST0027")]   # 비교 대상: 조석·고장 관측소 뺀 나머지
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for a, g_, title in [(ax[0], gap_leak, "[누수] 미래가 섞인 기준 (사상 전체 순위)"), (ax[1], gap, "[과거만] 지난 6시간 수위 변화 기준")]:
    a.hist(g_.reindex(is_gen).clip(-1, 1), bins=np.linspace(-1, 1, 51), color=C_NONTIDAL, alpha=.6, label="비조석 관측소")
    for v_ in g_[TIDAL]:
        a.axvline(min(v_, 1), color=C_TIDAL, lw=1.2)
    a.axvline(0, color="k", lw=.6)
    a.axvline(g_.reindex(is_gen).median(), color=C_NONTIDAL, ls="--")
    a.set_title(title, fontsize=10); a.set_xlabel("타깃 차이 (m) = 물 낮을 때 − 높을 때  (1m 넘는 조석은 오른쪽 끝)")
ax[0].plot([], [], color=C_TIDAL, label="조석 15곳"); ax[0].plot([], [], color=C_NONTIDAL, ls="--", label="비조석 중앙값")
ax[0].set_ylabel("관측소 수"); ax[0].legend(fontsize=8)
fig.tight_layout(); plt.show()
print(f"❌ 미래 섞인 기준 — 비조석 중앙값 {gap_leak.reindex(is_gen).median():+.3f} m  (조석이 아닌데도 +: 계산 방식이 만든 가짜 차이)")
print(f"✅ 과거만 쓴 기준 — 비조석 중앙값 {gap.reindex(is_gen).median():+.3f} m · 90% 지점 {gap.reindex(is_gen).quantile(.9):+.3f} m")
print(f"✅ 과거만 쓴 기준 — 조석 15곳 {gap[TIDAL].min():.2f} ~ {gap[TIDAL].max():.2f} m, 전부 양수: {(gap[TIDAL] > 0).all()}")

over = gap.reindex(is_gen)[gap.reindex(is_gen) > gap[TIDAL].min()].sort_values(ascending=False)
weak_ids = ["ST0151", "ST0193", "ST0226", "ST0229", "ST0260"]   # 5장에서 찾는 약한 조석
print(f"비조석 중 조석 최솟값({gap[TIDAL].min():.2f} m)을 넘는 곳: {len(over)}곳 →",
      [f"{s}{'(약한 조석)' if s in weak_ids else ''}" for s in over.index])


- 왼쪽(❌): 조석이 **아닌** 관측소도 중앙값이 **+0.07m**로 오른쪽에 쏠려 있어요. "사상에서 가장 낮은 순간이면 앞으로 오를 수밖에 없다"는 **계산 방식이 만든 가짜 차이**예요.
- 오른쪽(✅): 과거만 쓰면 비조석은 오히려 **음수(중앙값 −0.20m)** 예요. 비조석 하천은 "지난 6시간 오르고 있었으면 계속 오른다(추세 지속)"는 뜻이에요. 그런데 조석 15곳은 **전부 양수(0.35~6.8m)** 예요. **"내려왔으면 곧 오른다(되돌림)"** 는 거죠.
- 조석 15곳의 가장 작은 값(0.35m)도 비조석 90% 지점(+0.02m)보다 훨씬 커요. 비조석 중 0.35m를 넘는 곳은 **4곳뿐**이고, 그중 2곳(ST0226, ST0260)은 5장에서 볼 **약한 조석**이에요. 누수를 걷어 내니 두 집단이 오히려 **더 뚜렷하게 갈려요.**

이번엔 10칸 전체를 곡선으로 봐요. 곡선끼리 비교하려고 관측소마다 **타깃 표준편차로 나눴어요**.


In [ ]:

curve_z = curve.div(st.tstd, axis=0)          # 관측소 표준편차로 나눠 크기를 맞춤
fig, ax = plt.subplots(figsize=(9, 4))
g_c = curve_z.reindex(is_gen)
ax.fill_between(range(10), g_c.quantile(.1), g_c.quantile(.9), color=C_NONTIDAL, alpha=.2, label="비조석 10~90%")
ax.plot(range(10), g_c.median(), color=C_NONTIDAL, lw=2, label="비조석 중앙값")
for s_ in TIDAL:
    ax.plot(range(10), curve_z.loc[s_], color=C_TIDAL, lw=1, alpha=.8)
ax.plot([], [], color=C_TIDAL, label="조석 15곳 (한 줄 = 한 곳)")
ax.set_xticks(range(10)); ax.set_xticklabels(["6시간 새\n가장 많이\n내려옴"] + [""] * 8 + ["6시간 새\n가장 많이\n올라옴"])
ax.set_xlabel("지난 6시간 수위 변화 (dwl_6h, 관측소 안 10칸) — 과거 정보만 사용")
ax.set_ylabel("타깃 평균 ÷ 관측소 표준편차"); ax.axhline(0, color="k", lw=.5); ax.legend(fontsize=8)
ax.set_title("조석 15곳은 전부 '내려왔으면 곧 오른다', 비조석은 '오르던 게 계속 오른다'")
plt.show()


- 주황 선 15개가 **전부 왼쪽(내려와 있음)이 높고 오른쪽(올라와 있음)이 낮아요**. 파란 띠는 반대로 오른쪽이 올라가요.
- 다만 조차가 작은 관측소는 차이 자체가 작아요(0.35~0.9m). 방향은 15곳 모두 같지만, **조석 세기는 관측소마다 연속적으로 다르다**는 신호예요. 이건 5장에서 이어서 봐요.

### 4-7. 조석 15곳이 차지하는 몫

In [ ]:
row_share = df.station_id.isin(TIDAL).mean()
sse_share = st.loc[TIDAL, "sse_share"].sum()
top10 = st.sort_values("sse_share", ascending=False).head(10)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})
for i, (lab, v) in enumerate([("행 수", row_share), ("0 예측 제곱오차", sse_share)]):
    ax[0].barh(i, v * 100, color=C_TIDAL); ax[0].barh(i, 100 - v * 100, left=v * 100, color=C_ALL, alpha=.4)
    ax[0].text(v * 100 + 1, i, f"{v:.1%}", va="center")
ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(["행 수", "0 예측\n제곱오차"]); ax[0].set_xlabel("%")
ax[0].set_title("조석 15곳(주황)의 몫")
tt = st.loc[TIDAL].sort_values("sse_share", ascending=False)
ax[1].bar(range(15), tt.sse_share * 100, color=C_TIDAL)
ax[1].axhline(100 / len(st), color="k", ls=":", lw=.8); ax[1].text(14.4, 100 / len(st) + .3, "관측소 평균 몫", ha="right", fontsize=8)
ax[1].set_xticks(range(15)); ax[1].set_xticklabels(tt.index, rotation=60, fontsize=7); ax[1].set_ylabel("제곱오차 비중 (%)")
ax[1].set_title("15곳 각각의 오차 비중")
fig.tight_layout(); plt.show()
print(f"오차 상위 10곳 중 조석: {top10.index.isin(TIDAL).sum()}곳")
print(f"타깃 표준편차 중앙값: 조석 {st.loc[TIDAL, 'tstd'].median():.2f} m vs 나머지 {st.loc[~st.is_tidal, 'tstd'].median():.2f} m")

**행은 3.2%인데 오차는 51%**예요. 오차 상위 10곳 중 8곳이 조석 관측소예요.
다만 15곳이 다 크게 기여하는 건 아니고, **진폭이 큰 앞쪽 5~8곳이 대부분**을 만들어요. 뒤쪽 몇 곳은 관측소 평균 몫(0.2%)과 비슷해요.

2장의 산점도를 이번엔 조석 15곳에 색을 칠해서 다시 볼게요.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(st.tstd[~st.is_tidal], st.tmean[~st.is_tidal], s=12, color=C_NONTIDAL, alpha=.6, label=f"나머지 n={(~st.is_tidal).sum()}")
ax.scatter(st.tstd[st.is_tidal], st.tmean[st.is_tidal], s=30, color=C_TIDAL, label="조석 15곳 (자기상관으로 판정)")
ax.set_xscale("log"); ax.set_xlabel("관측소별 타깃 표준편차 (m, log)"); ax.set_ylabel("관측소별 타깃 평균 (m)")
ax.legend(loc="upper left"); ax.set_title("떨어져 나온 무리는 전부 조석. 하지만 조석 전부가 떨어져 나온 건 아니다")
plt.show()

떨어져 나온 무리(7곳)는 **전부 조석**이에요. 그런데 나머지 조석 8곳은 일반 관측소 무리의 위쪽 가장자리에 섞여 있어요.
→ **평균·표준편차만 보고는 조석 관측소를 다 못 찾아요.** 주기(자기상관)로 찾아야 해요.

## 5. 조석은 "있다/없다"가 아니라 "세기"다

기준(0.6)은 사람이 정한 문턱이에요. 문턱 바로 아래에 **약하게 조석을 타는 관측소**가 있을 수 있고, 12시간이 아닌 **다른 주기**로 움직이는 관측소도 있을 수 있어요.
그래서 **12시간 자기상관**과 **24시간 자기상관**을 두 축으로 놓고 465곳을 전부 찍어요.

- 조석: 12h도 +, 24h도 + (24시간 ≈ 조석 두 번)
- 하루 한 번 사람이 조작하는 곳(수문 개폐, 발전 방류 등): 24h만 +, 12h는 0이나 −

In [ ]:
strong = st.is_tidal
weak = (~strong) & st.acf12.between(0.35, 0.6) & (st.acf6 < -0.2)     # 문턱 아래의 약한 파도
daily = (~strong) & (st.acf24 >= 0.4) & (st.acf12 < 0.15)             # 24시간만 있는 곳
WEAK_TIDAL, OPERATED_24H = st.index[weak].tolist(), st.index[daily].tolist()
st["group"] = np.select([strong, weak, daily], ["조석", "약한 조석", "24h 주기"], default="일반")
GCOL = {"일반": C_NONTIDAL, "조석": C_TIDAL, "약한 조석": C_WEAK, "24h 주기": C_OPER}
print("약한 조석:", WEAK_TIDAL)
print("24h 주기:", OPERATED_24H)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for k, col in GCOL.items():
    m = st.group == k
    ax[0].scatter(A.loc[m, 12], A.loc[m, 24], s=8 if k == "일반" else 30, color=col,
                  alpha=.35 if k == "일반" else 1, label=f"{k} (n={int(m.sum())})")
for sid in OPERATED_24H:
    ax[0].annotate(sid, (A.loc[sid, 12], A.loc[sid, 24]), fontsize=8, xytext=(4, 3), textcoords="offset points")
ax[0].axhline(0, color=C_ALL, lw=.6); ax[0].axvline(0, color=C_ALL, lw=.6)
ax[0].set_xlabel("12h 자기상관"); ax[0].set_ylabel("24h 자기상관"); ax[0].legend(fontsize=8, loc="lower right")
ax[0].set_title("주기성 지도")
# 그룹별 자기상관 곡선: 각 관측소 한 줄씩
for k in ["조석", "약한 조석", "24h 주기"]:
    for s_ in st.index[st.group == k]:
        ax[1].plot(A.columns, A.loc[s_], color=GCOL[k], lw=1, alpha=.8)
    ax[1].plot([], [], color=GCOL[k], label=k)
ax[1].plot(A.columns, A.loc[st.group == "일반"].median(), color=C_NONTIDAL, lw=2, label="일반 중앙값")
ax[1].axhline(0, color="k", lw=.5); ax[1].set_xlabel("시차 (h)"); ax[1].legend(fontsize=8)
ax[1].set_title("그룹별 자기상관 곡선 (한 줄 = 한 관측소)")
fig.tight_layout(); plt.show()
st.groupby("group").agg(관측소수=("tmean", "size"), acf12_중앙값=("acf12", "median"), acf24_중앙값=("acf24", "median"),
                        주기_중앙값=("period", "median"), 강우반응ρ=("rain_rho", "median"), 오차비중합=("sse_share", "sum")).round(3)

**지도에서 보이는 것**
- 🟠 **조석 15곳**은 오른쪽 위 끝에 있어요.
- 🟢 **약한 조석 5곳**(ST0151, ST0193, ST0226, ST0229, ST0260)은 문턱 바로 아래에서 같은 파도를 약하게 그려요. 5곳 모두 주기가 12.0~12.6시간이에요. 바다에서 조금 더 먼 감조 구간일 거라고 추정해요.
- 🩷 **24h 주기 2곳**(ST0194, ST0200)은 12시간 봉우리가 없고 24시간 봉우리만 있어요. 강우 반응 ρ도 −0.04로 **비와 전혀 무관**해요. 수문을 매일 여닫는 식의 **사람의 조작**으로 보여요.
- 일반 관측소도 조석 쪽으로 **연속적으로 퍼져 있어요**. 딱 잘리는 경계가 없어요.

→ 그래서 모델에 "조석이다/아니다"를 0/1로 넣기보다 **12h·24h 자기상관 값 자체를 관측소 특성으로** 넣는 게 정보를 덜 버려요.

약한 조석 5곳과 24h 주기 2곳도 **전부** 수위를 그려서 확인해요.

In [ ]:
show = [(s_, "약한 조석") for s_ in WEAK_TIDAL] + [(s_, "24h 주기") for s_ in OPERATED_24H]
fig, axes = plt.subplots(len(show), 1, figsize=(12, 1.7 * len(show)), sharex=True)
for ax, (s_, k) in zip(axes, show):
    d = df[df.station_id == s_]
    blk = d.groupby("event_block").size().idxmax()
    d = d[d.event_block == blk].sort_values("hour").iloc[:120]
    h = d.pos_in_block - d.pos_in_block.min()
    ax.plot(h, d.wl_t, color=GCOL[k], lw=1)
    a2 = ax.twinx(); a2.bar(h, d.sfc_rain_1h, color=C_RAIN, alpha=.5, width=1)
    a2.invert_yaxis(); a2.set_ylim(max(d.sfc_rain_1h.max() * 3, 5), 0); a2.grid(False); a2.set_yticks([])
    ax.set_title(f"{s_} ({k})  12h {A.loc[s_, 12]:+.2f} · 24h {A.loc[s_, 24]:+.2f} · ρ {st.loc[s_, 'rain_rho']:+.2f}", fontsize=8)
    ax.tick_params(labelsize=7)
axes[-1].set_xlabel("가장 긴 사상 안 경과 시간 (h, 앞 5일)")
fig.suptitle("약한 조석 5곳(초록) + 24h 주기 2곳(분홍) 전부", x=.02, ha="left")
fig.tight_layout(); plt.show()

약한 조석은 **작은 톱니**가 보여요. 24h 주기 관측소는 톱니가 아니라 **하루 한 번 계단처럼** 올라갔다 내려가요.

앞으로 "비조석"은 **조석 15곳과 고장 난 관측소 1곳(ST0027, 수위가 한 번도 안 변함)을 뺀 나머지**를 뜻해요(보고서 기준과 같아요). 약한 조석 5곳은 오차 비중이 0.5%뿐이라 비조석에 남겨 뒀어요.

## 6. 조석이 분석을 어떻게 흐리나 — 조석을 빼고 다시 보기

조석 15곳이 오차의 절반이면, **전체 데이터로 본 관계에도 조석이 섞여서** 엉뚱한 결론이 나올 수 있어요. 타깃이 들어가는 분석을 **전체(회색)와 비조석(파랑)으로 나란히** 다시 그려 볼게요.

### 6-1. "물이 확 빠진 직후에 오히려 크게 오른다"는 착시

In [ ]:
STUCK = ["ST0027"]
df["is_tidal"] = df.station_id.isin(TIDAL)
nontidal = df[~df.is_tidal & ~df.station_id.isin(STUCK)]
GROUPS = [("전체", df, C_ALL), ("비조석", nontidal, C_NONTIDAL)]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
res = {}
for a, x in zip(ax, ["dwl_1h", "dwl_3h"]):
    for g, d, c in GROUPS:
        edges = np.unique(np.quantile(d[x], np.linspace(0, 1, 41)))      # 40칸(각 2.5%)
        k = pd.cut(d[x], edges, include_lowest=True)
        b = d.groupby(k, observed=True).agg(x=(x, "median"), m=(TARGET, "mean"))
        a.plot(b.x, b.m, "-o", ms=2.5, color=c, label=g)
        res[(x, g)] = {"가장 많이 떨어진 칸": b.m.iloc[0], "가장 많이 오른 칸": b.m.iloc[-1]}
    symlog_x(a, .01); a.set_xlabel(f"{x}: 최근 수위 변화 (m, symlog)"); a.axvline(0, color=C_ALL, lw=.6)
    a.annotate("← 확 빠진 직후", xy=(.02, .9), xycoords="axes fraction", fontsize=8)
    a.annotate("계속 오르는 중 →", xy=(.72, .9), xycoords="axes fraction", fontsize=8)
ax[0].set_ylabel("타깃 평균 (m)"); ax[0].legend()
fig.suptitle("전체로 보면 V자, 비조석만 보면 왼쪽 가지가 사라진다", x=.02, ha="left")
fig.tight_layout(); plt.show()
pd.DataFrame(res).T.round(3)

- 전체(회색)는 **V자**예요. 오르는 중이면 계속 오르고(오른쪽), **확 빠진 직후에도** 타깃이 커요(왼쪽, 평균 0.32m).
- 비조석(파랑)은 왼쪽 가지가 **0.02m로 사라져요**. 오른쪽 가지는 거의 그대로예요.

**그럼 왼쪽 가지는 전부 조석에서 오는 걸까요?** 전체 데이터에서 "가장 많이 떨어진 2.5%" 행만 모아서, 그 타깃이 어느 관측소에서 오는지 나눠 봐요.

In [ ]:
q = df.dwl_1h.quantile(.025)                               # 전체 기준 가장 많이 떨어진 2.5%
b = df[df.dwl_1h <= q]
contrib = b.groupby("station_id")[TARGET].sum().sort_values(ascending=False)
print(f"이 구간 행: {len(b):,}개 · 그중 조석 관측소 행 비중 {b.is_tidal.mean():.0%}")
print(f"이 구간 타깃 합 중 조석 관측소 몫: {b.loc[b.is_tidal, TARGET].sum() / b[TARGET].sum():.0%}")

fig, ax = plt.subplots(figsize=(10, 3.4))
top = contrib.head(20)
ax.bar(range(len(top)), top, color=[C_TIDAL if s_ in TIDAL else C_NONTIDAL for s_ in top.index])
ax.set_xticks(range(len(top))); ax.set_xticklabels(top.index, rotation=60, fontsize=7)
ax.set_ylabel("타깃 합 (m)"); ax.set_title("'확 빠진 직후' 구간의 타깃은 거의 전부 조석 관측소에서 (주황 = 조석)")
plt.show()

그 구간 행의 41%만 조석인데, **타깃 합은 사실상 100%가 조석 관측소 몫**이에요(비조석 행의 타깃은 더하면 0 근처예요). 그러니까 V자의 왼쪽 가지는 **조석이 만든 착시**예요.

⚠️ **그런데 조심할 점이 있어요.** "빠진 직후 반등"이 **15곳 모두의 공통 성질은 아니에요.** 관측소마다 **자기 기준으로** 가장 빠르게 빠지는 5% 순간을 골라 보면 결과가 갈려요.

In [ ]:
# 관측소마다 '자기 dwl_1h 하위 5%' 순간의 타깃 평균
thr = df.groupby("station_id").dwl_1h.transform(lambda v: v.quantile(.05))
own_fast_fall = df[df.dwl_1h <= thr].groupby("station_id")[TARGET].mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.5, 1]})
v = own_fast_fall[TIDAL]
ax[0].bar(range(15), v, color=np.where(v > 0, C_TIDAL, C_ALL))
ax[0].axhline(0, color="k", lw=.5); ax[0].set_xticks(range(15)); ax[0].set_xticklabels(TIDAL, rotation=60, fontsize=7)
ax[0].set_ylabel("타깃 평균 (m)"); ax[0].set_title("조석 15곳: '자기 기준 가장 빨리 빠질 때' 타깃")
# 대비: 4-6의 '지난 6시간 한참 내려와 있을 때' 기준(과거만 사용)은 15곳 모두 양수
ax[1].scatter(own_fast_fall[TIDAL], gap[TIDAL], s=30, color=C_TIDAL)
ax[1].axvline(0, color="k", lw=.5); ax[1].axhline(0, color="k", lw=.5)
ax[1].set_xlabel("가장 빨리 빠질 때 기준 (m)"); ax[1].set_ylabel("6시간 새 한참 내려와 있을 때 기준 (4-6, m)")
ax[1].set_title("'한참 내려와 있을 때'는 15곳 모두 +, '가장 빨리 빠질 때'는 제각각")
fig.tight_layout(); plt.show()
print(f"'가장 빨리 빠질 때' 타깃이 양수인 조석 관측소: {(v > 0).sum()}/15")

- 15곳 중 **6곳만** 양수예요. 몇 곳(ST0358, ST0326, ST0415 등)은 가장 빨리 빠진 직후 크게 반등해요. 반대로 ST0292처럼 **더 내려가는 곳**도 있어요.
- 이유는 4-4절에서 본 것처럼 **하구 조석이 비대칭**이기 때문이에요. 강에서는 밀물이 짧고 가파르게, 썰물이 길고 완만하게 오는 경우가 많아요. 그래서 "가장 빠르게 빠지는 순간"이 썰물의 끝인지 중간인지가 관측소마다 달라요.
- 반면 4-6의 **"지난 6시간 동안 한참 내려와 있으면 타깃이 크다"는 15곳 모두**에서 성립해요. 6시간은 조석 반 주기라서, 이 값이 조석의 **위치(위상)** 를 알려 주기 때문이에요.

→ **조석 관측소를 설명하는 공통 규칙은 "하강 속도"가 아니라 "지금 조석의 어느 위치(위상)에 있나"예요.** 모델에 줄 변수도 `wl_t − wl_t−12h`(한 주기 전 대비 현재 높이)나 `wl_t − wl_t−6h`(반 주기 전 대비)처럼 **위상을 알려 주는 것**이 맞아요.

### 6-2. 비가 안 올 때의 흔들림도 대부분 조석

In [ ]:
for g, d, _ in GROUPS:
    dry = d[d.sfc_rain_1h == 0]
    print(f"{g:4s} 비 안 올 때 타깃 표준편차 = {dry[TARGET].std():.3f} m")

# 관측소 단위로도 확인: 비 안 올 때 타깃 표준편차
dry_std = df[df.sfc_rain_1h == 0].groupby("station_id")[TARGET].std()
fig, ax = plt.subplots(figsize=(10, 3.2))
bins = np.logspace(-3, .5, 40)
ax.hist(dry_std[st.group.reindex(dry_std.index) == "일반"], bins=bins, color=C_NONTIDAL, alpha=.6, label="일반")
ax.hist(dry_std[WEAK_TIDAL], bins=bins, color=C_WEAK, label="약한 조석")
for v_ in dry_std[TIDAL]:
    ax.axvline(v_, color=C_TIDAL, lw=1.2)
ax.plot([], [], color=C_TIDAL, label="조석 15곳")
ax.set_xscale("log"); ax.set_xlabel("비 안 올 때 타깃 표준편차 (m, log)"); ax.set_ylabel("관측소 수"); ax.legend(fontsize=8)
ax.set_title("비가 안 와도 크게 흔들리는 관측소 = 대부분 조석"); plt.show()
print(f"일반 관측소 99% 지점: {dry_std[st.group.reindex(dry_std.index) == '일반'].quantile(.99):.2f} m")
print("조석 15곳:", dry_std[TIDAL].round(2).tolist())

비가 안 올 때 타깃의 흔들림이 전체 0.33m → 비조석 0.17m로 **절반**이 돼요.
관측소 단위로 보면 조석 15곳 중 **9곳은 일반 관측소 99% 지점(0.43m)보다도 커요**. 나머지 조석 6곳(0.16~0.42m)은 일반 관측소 상위권과 겹쳐요. 여기서도 **조석은 세기가 제각각**이라는 게 다시 보여요.

### 6-3. "관측소마다 다르다"는 착시 — 분산 분해

**비유**: 전교생 시험 점수가 들쭉날쭉한 이유를 쪼개 본다고 해 봐요.
- **학교 차이**(어느 학교냐) / **시험 회차 차이** / **학교×회차** / **같은 회차 안에서 학생마다 다른 부분**

여기선 학교 = 관측소, 회차 = 사상, 학생 = 사상 안의 시각이에요. 타깃이 들쭉날쭉한 이유 중 **"어느 관측소냐"가 몇 %** 인지 보는 거예요.

In [ ]:
def var_decomp(d):
    """타깃 분산을 관측소 / 사상 / 관측소×사상 / 사상 안 시간 변동으로 쪼갠다"""
    y = d[TARGET].astype(float); mu = y.mean(); tot = ((y - mu) ** 2).sum()
    s_mean = d.groupby("station_id")[TARGET].transform("mean")                    # 관측소 평균
    e_mean = d.groupby("event_block")[TARGET].transform("mean")                   # 사상 평균
    se_mean = d.groupby(["station_id", "event_block"])[TARGET].transform("mean")  # 관측소×사상 평균
    a = ((s_mean - mu) ** 2).sum() / tot
    b = ((e_mean - mu) ** 2).sum() / tot
    c = ((se_mean - mu) ** 2).sum() / tot
    return {"관측소": a, "사상": b, "관측소×사상": max(c - a - b, 0), "사상 안 시간 변동": 1 - c}

T = pd.DataFrame({g: var_decomp(d) for g, d, _ in GROUPS})
fig, ax = plt.subplots(figsize=(10, 2.8))
left = np.zeros(2)
for (k, row), col in zip(T.iterrows(), ["#2a78d6", "#eda100", "#1baf7a", "#d9d8d2"]):
    ax.barh(range(2), row.values * 100, left=left, color=col, label=k)
    for i, v_ in enumerate(row.values):
        if v_ > .04: ax.text(left[i] + v_ * 50, i, f"{v_:.0%}", ha="center", va="center", fontsize=9)
    left += row.values * 100
ax.set_yticks(range(2)); ax.set_yticklabels(T.columns); ax.set_xlabel("타깃 분산 중 비율 (%)")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(.5, -.35))
ax.set_title("조석을 빼면 '관측소' 몫이 18% → 2%"); plt.show()
(T * 100).round(1)

전체로 보면 "어느 관측소냐"가 18%나 설명하는 것 같지만, 조석을 빼면 **2%** 예요. 18% 대부분이 조석 관측소 몇 곳이 유별나서 생긴 거예요.
비조석에서 가장 큰 몫은 **"같은 사상 안에서 시각마다 달라지는 부분"(74%)** 이에요.

→ 비조석 관측소를 맞히는 열쇠는 "어느 관측소냐"가 아니라 **"지금 비가 얼마나 왔고, 물이 오르는 중이냐"** 같은 **지금 이 순간의 상황**이에요.

## 7. 모델은 조석을 맞힐 수 있나?

조석의 위상 정보는 이미 입력에 들어 있어요. `wl_t`와 1·3·6·12·24시간 전 수위가 주어지니까요.
- 예: `wl_t − wl_t−6h`가 크게 음수 → 지난 6시간 썰물 → 곧 밀물 → 타깃 클 것
- 예: `wl_t − wl_t−12h`가 0 근처 → 한 주기 전과 같은 위치

그러면 트리 모델(HGB)이 이 규칙을 스스로 배울 수 있을까요?

**검증 방법 (GroupKFold, 사상 단위)**: 같은 사상의 데이터가 학습과 검증에 동시에 들어가면 **컨닝**이 돼요. 바로 옆 시각의 값이 거의 같으니까요. 그래서 사상을 통째로 5묶음으로 나눠, 한 묶음씩 빼고 학습한 뒤 뺀 묶음을 예측해요.

HGB 학습은 몇 분 걸려서, 원본 `src/s05_baseline.py` 결과(`cache/results/s05_baseline.csv`)를 읽어 와요. 직접 돌려 보려면 `RUN_BASELINE = True`로 바꾸면 돼요.

In [ ]:
RUN_BASELINE = False

if RUN_BASELINE:
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.model_selection import GroupKFold
    FEATURES = ["wl_t", "wl_t_minus_1h", "wl_t_minus_3h", "wl_t_minus_6h", "wl_t_minus_12h", "wl_t_minus_24h",
                "imerg_rain_1h", "imerg_rain_6h", "imerg_rain_24h", "sfc_rain_1h", "sfc_rain_6h", "sfc_rain_24h",
                "radar_reflectivity_mean", "radar_reflectivity_p95", "radar_reflectivity_ge20_fraction",
                "cat_area", "wsarea", "mean_elevation", "mean_slope", "urban_ratio", "forest_ratio", "stream_order"]
    DWL = ["dwl_1h", "dwl_3h", "dwl_6h", "dwl_12h", "dwl_24h"]
    full = pd.read_parquet(CACHE / "train_sorted.parquet", columns=["station_id", "station_code", "event_block", TARGET] + FEATURES + DWL)
    full["is_tidal"] = full.station_id.astype(str).isin(TIDAL)
    yy = full[TARGET].to_numpy(float)
    rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
    rows = []
    for name, feats, stat in [("0 예측", None, False), ("HGB 원변수", FEATURES, False), ("HGB 원변수+차분+관측소통계", FEATURES + DWL, True)]:
        oof = np.zeros(len(full))                          # out-of-fold 예측: 각 행은 자기를 안 본 모델이 예측
        for tri, vai in GroupKFold(5).split(full, groups=full.event_block):
            if feats is None: continue
            tr, va, f = full.iloc[tri], full.iloc[vai], list(feats)
            if stat:  # 관측소 타깃 통계는 반드시 학습 묶음 안에서만 계산 (누수 방지)
                g = tr.groupby("station_code")[TARGET].agg(st_mean="mean", st_std="std", st_q95=lambda s: s.quantile(.95))
                tr, va, f = tr.join(g, on="station_code"), va.join(g, on="station_code"), f + ["st_mean", "st_std", "st_q95"]
            m = HistGradientBoostingRegressor(max_iter=300, learning_rate=.1, max_leaf_nodes=63, min_samples_leaf=50, random_state=0)
            oof[vai] = m.fit(tr[f], tr[TARGET]).predict(va[f])
        t_ = full.is_tidal.to_numpy()
        rows.append({"method": name, "rmse": rmse(yy, oof), "rmse_tidal": rmse(yy[t_], oof[t_]), "rmse_nontidal": rmse(yy[~t_], oof[~t_])})
    base = pd.DataFrame(rows)
else:
    base = pd.read_csv(CACHE / "results" / "s05_baseline.csv")
    base = base[base.split.str.startswith("Group")].drop(columns=["split", "sec"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 3.8))
xx = np.arange(len(base))
ax.bar(xx - .2, base.rmse_tidal, .4, color=C_TIDAL, label="조석 15곳 RMSE")
ax.bar(xx + .2, base.rmse_nontidal, .4, color=C_NONTIDAL, label="나머지 RMSE")
for i, r in base.iterrows():
    ax.text(i - .2, r.rmse_tidal + .03, f"{r.rmse_tidal:.2f}", ha="center", fontsize=7)
    ax.text(i + .2, r.rmse_nontidal + .03, f"{r.rmse_nontidal:.2f}", ha="center", fontsize=7)
ax.set_xticks(xx); ax.set_xticklabels(base.method, fontsize=8); ax.set_ylabel("RMSE (m)"); ax.legend()
ax.set_title("단순한 방법 → HGB로 갈수록 조석 오차가 확 줄어든다")
plt.show()
base.round(4)

- 0으로 찍으면 조석 RMSE가 **1.64m**, 관측소 평균으로 찍어도 1.32m예요.
- HGB를 쓰면 **0.45m → 0.38m**까지 떨어져요. 비조석은 0.29 → 0.22로, 상대적으로 덜 줄어요.

이것도 **15곳 전부**에서 그런지 봐야겠죠. 원본 `src/s09_network.py`가 저장해 둔 최고 모델(HGB 원변수+차분+관측소통계, 사상 단위 교차검증)의 예측값으로, **관측소마다** "모델 RMSE ÷ 0 예측 RMSE"를 구해요. 작을수록 모델이 많이 개선한 거예요.

In [ ]:
oof = pd.read_parquet(CACHE / "oof_s09.parquet")[["station_code", "hour", "oof_base"]]
m = df[["station_code", "hour", "station_id", TARGET]].merge(oof, on=["station_code", "hour"])
print(f"불러온 예측의 전체 RMSE: {np.sqrt(((m[TARGET] - m.oof_base) ** 2).mean()):.4f} (보고서 0.2264와 같아야 함)")

per = m.groupby("station_id").apply(lambda z: pd.Series({
    "rmse_0": np.sqrt((z[TARGET] ** 2).mean()),                       # 0으로 찍었을 때
    "rmse_model": np.sqrt(((z[TARGET] - z.oof_base) ** 2).mean())}))  # 모델
per["ratio"] = per.rmse_model / per.rmse_0
m["e2"] = (m[TARGET] - m.oof_base) ** 2
print(f"모델 기준 조석 15곳의 제곱오차 비중: {m.loc[m.station_id.isin(TIDAL), 'e2'].sum() / m.e2.sum():.1%}  (0 예측일 땐 51%)")

fig, ax = plt.subplots(figsize=(10, 3.4))
gen = per.loc[st.group.reindex(per.index) == "일반", "ratio"]
ax.hist(gen, bins=np.linspace(0, 1.2, 49), color=C_NONTIDAL, alpha=.6, label="일반 관측소")
for v_ in per.loc[TIDAL, "ratio"]:
    ax.axvline(v_, color=C_TIDAL, lw=1.2)
ax.plot([], [], color=C_TIDAL, label="조석 15곳")
ax.axvline(gen.median(), color=C_NONTIDAL, ls="--"); ax.text(gen.median() + .01, ax.get_ylim()[1] * .9, "일반 중앙값", fontsize=8, color=C_NONTIDAL)
ax.set_xlabel("모델 RMSE ÷ 0 예측 RMSE  (작을수록 모델이 잘 맞힘)"); ax.set_ylabel("관측소 수"); ax.legend(fontsize=8)
ax.set_title("조석 15곳 모두 일반 관측소 중앙값 이하")
plt.show()
print("조석 15곳 비율:", per.loc[TIDAL, "ratio"].round(2).tolist())
print(f"일반 관측소 비율 중앙값: {gen.median():.2f}")

- 조석 15곳은 비율이 **0.14~0.75**로, **전부 일반 관측소 중앙값(0.76) 이하**예요. 특히 진폭이 큰 곳(ST0358, ST0292, ST0326)은 오차가 **1/6 수준**으로 줄어요.
- 모델 기준으로는 조석 15곳의 오차 비중이 51% → **8.8%**로 떨어져요.

**왜 조석은 잘 맞힐까요?** 조석은 **달이 정해 주는 규칙적인 움직임**이라서, 지난 수위 몇 개만 보면 다음이 거의 정해져요. 반면 비 때문에 불어나는 물은 "비가 얼마나 더 올지", "언제 도착할지"가 불확실해요.

→ **조석은 0 예측 기준으로는 오차의 절반이었지만, 모델을 쓰면 대부분 풀려요. 이제 남은 오차의 주인공은 비조석 관측소의 큰 호우예요.**

(참고: 이웃 관측소 정보(연결망 변수)를 넣으면 비조석은 2.9% 좋아지는데 조석은 2.7% 나빠졌어요(`insights.md` 2-2). 조석은 따로 다뤄야 한다는 근거가 하나 더 있는 셈이에요.)

## 8. 정리 — 한눈에 보기

### 무엇을 알아냈나

| # | 쉽게 말하면 | 집단 전체에 해당? | 근거 |
|---|---|---|---|
| 1 | 바다 가까운 **15곳**은 비가 아니라 밀물·썰물로 수위가 오르내린다 | ✅ 15곳 전부 자기상관 파도 모양, 주기 12.2~12.4h | 4-1 ~ 4-4 |
| 2 | 이 15곳은 비와 무관하다 | ✅ 15곳 전부 ρ = −0.03~+0.12 (일반 중앙값 +0.42) | 4-5 |
| 3 | 6시간 새 한참 내려와 있으면(썰물 끝 근처) 곧 오른다. 비조석은 반대로 오르던 게 계속 오른다 | ✅ 15곳 전부 (0.35~6.8m), 과거 정보만으로 확인 | 4-6 |
| 4 | 데이터 3.2%가 0 예측 오차의 51% | ⚠️ 집단 합계. 주로 진폭 큰 5~8곳이 만든다 | 4-7 |
| 5 | 조석은 있다/없다가 아니라 세기의 연속체 (강 15 + 약 5), 24시간 조작 2곳은 따로 | ✅ 5곳·2곳 전부 수위로 확인 | 5 |
| 6 | "빠진 직후 반등(V자)"은 조석이 만든 착시 | ✅ 그 구간 타깃 합 ≈100% 조석. ⚠️ 관측소별 '가장 빠른 하강 직후'는 15곳 중 6곳만 반등 → 공통 규칙은 **위상** | 6-1 |
| 7 | "관측소마다 다르다(18%)"도 대부분 조석 때문. 비조석은 2% | 집단 비교 | 6-3 |
| 8 | 모델은 조석을 꽤 잘 맞힌다. 남은 문제는 비조석 큰 호우 | ✅ 15곳 전부 개선 비율이 일반 중앙값 이하 | 7 |

### 모델링할 때 할 일

1. **평가**: 전체 RMSE만 보지 말고 **비조석 RMSE를 따로** 본다. 조석 몇 곳이 순위를 흔들 수 있다.
2. **관측소 특성**: "조석이냐 0/1" 대신 **12h·24h 자기상관 값**을 넣는다. 학습 묶음 안에서 계산한다.
3. **위상 변수**: `wl_t − wl_t−12h`, `wl_t − wl_t−6h`처럼 "지금 조석의 어느 위치인가"를 알려 주는 차분을 넣는다.
4. **24시간 주기 2곳**: 비 변수는 쓸모없다. `wl_t_minus_24h`(어제 같은 시각)가 핵심이다.
5. **개선의 무게중심**: 조석은 이미 많이 풀렸으니, 비조석의 **큰 상승 국면**(비가 오고 물이 오르는 중)에 집중한다.

### 누수 점검표 — 이 노트북에서 미래 정보를 쓴 곳

| 어디 | 미래 정보를 썼나? | 판단 |
|---|---|---|
| 1장 타깃 재계산 | ✅ 썼음 (1~6시간 뒤 수위) | 정의 확인용이에요. test에도 뒤 시각 행이 있어서 똑같이 하면 타깃이 계산돼요. **모델·제출에는 절대 사용 금지** |
| 4장 자기상관·주기, 5장 주기성 지도 | 관측소의 train 기간 전체 수위 변화 | 타깃은 안 써요. 관측소 고유 특성이에요. 모델에 넣을 땐 **train(학습 묶음) 안에서만** 계산해요 |
| 4-6 물 높이 기준 | ❌ 처음엔 사상 전체 순위(미래 포함) → 비조석까지 +0.07m 부풀림 | **과거만 쓰는 `dwl_6h`로 교체.** 결론은 유지되고 더 선명해졌어요 |
| 조석 15곳 목록 | train 전체로 판정 | 결과를 나눠 보는 용도(층화)로만 써요. 7장 모델 입력에는 없어요 |
| 6장 V자·분산 분해 | 입력은 과거 값(`dwl_*`), 타깃은 결과 쪽 | 관계를 설명하는 그림일 뿐, 입력에 미래가 없어요 |
| 7장 모델 | 입력 = 과거 수위·강우·유역 + 관측소 타깃 통계(학습 묶음 안에서 계산) | **누수 없음.** 사상 단위 GroupKFold로 검증했어요 |

> 🚫 test 안의 미래 행은 절대 쓰지 않는다. 관측소 통계(타깃 통계, 자기상관)는 학습 묶음 안에서만 계산한다.